# 5. spaCy II: Pipelines clíniques amb CARMEN-I

Tractament de Dades Textuals i Codificades — Eixample Clínic, 2026

Pol Pastells, ppastells@eixampleclinic.es

---

A aquesta sessió construirem una pipeline clínica completa sobre textos reals del corpus **CARMEN-I**.

## 0. Càrrega del corpus i eines

Comencem carregant tot el que necessitem. Fes servir els textos de la carpeta **`replaced`**: és la versió on les dades sensibles han estat substituïdes. Més endavant, a l'apartat 4, compararem amb la carpeta **`masked`**.

In [ ]:
# Si cal, descarrega el model:
# pip install spacy
# ! python -m spacy download es_core_news_sm

import re
from pathlib import Path

import pandas as pd
import spacy
from spacy import displacy

# Model general d'espanyol
nlp_es = spacy.load("es_core_news_sm")
print("Model carregat:", nlp_es.meta["name"])
print("Components de la pipeline:", nlp_es.pipe_names)

In [ ]:
# ── Rutes al corpus ──────────────────────────────────────────────────────────
# Ajusta CORPUS_ROOT a on tinguis el corpus descarregat
CORPUS_ROOT = Path("Corpus CARMEN/CARMEN1_v1")
TXT_REPLACED = CORPUS_ROOT / "txt" / "replaced"
TXT_MASKED = CORPUS_ROOT / "txt" / "masked"
TSV_REPLACED_NER = CORPUS_ROOT / "tsv" / "replaced" / "CARMEN-I_replaced_ner.tsv"
TSV_REPLACED_ANON = CORPUS_ROOT / "tsv" / "replaced" / "CARMEN-I_replaced_anon.tsv"
TSV_MASKED_ANON = CORPUS_ROOT / "tsv" / "masked" / "CARMEN-I_masked_anon.tsv"

In [ ]:
# Carregar les anotacions
df = pd.read_csv(TSV_REPLACED_NER, sep="\t")
df

In [ ]:
# Quantes files i columnes tenim?
print(df.shape)
print(df.columns.tolist())

In [ ]:
# Quines categories d'entitats hi ha?
df.tag.value_counts()

In [ ]:
# Visualitzem-ho com a gràfic de barres
df["tag"].value_counts().plot(kind="bar")

In [ ]:
# hi ha algun text problemàtic? Comprova què passa
df[df.text.isna()]

### Solució

In [ ]:
# Pandas llegeix un parell d'anotacións de NA (sodi) com a NaN
# ho podem remediar amb:
df = pd.read_csv(TSV_REPLACED_NER, sep="\t", keep_default_na=False)

## Seguim

In [ ]:
fitxers = df.name.unique()

In [ ]:
# Tots els fàrmacs del corpus
farmacs = df[df["tag"] == "FARMACO"]
print(f"Total fàrmacs: {len(farmacs)}")
farmacs["text"].value_counts().head(20)

### Exercici 1
- Quina és la malaltia més freqüent al corpus?
- Quants símptomes hi ha en total?
- Quin fitxer té més anotacions?

In [ ]:
# ── Fitxer de metadades ───────────────────────────────────────────────────────
mappings = pd.read_csv(
    CORPUS_ROOT / "CARMEN1_mappings.tsv",
    sep="\t",
    names=["name", "lang", "has_annotations"],
    header=0,
)
print(f"Total de fitxers: {len(mappings)}")
print(f"  Espanyol: {(mappings.lang == 'es').sum()}")
print(f"  Català:   {(mappings.lang == 'cat').sum()}")
print(f"  Bilingüe: {(mappings.lang == 'bi').sum()}")
print(f"  Anotats:  {mappings.has_annotations.sum()}")
mappings.head()

## 1. El model general no sap (massa) medicina

El model `es_core_news_sm` ha estat entrenat sobre textos periodístics i web generals. Està dissenyat per detectar **PER** (persones), **ORG** (organitzacions) i **LOC** (llocs). Els textos clínics de CARMEN-I contenen entitats molt diferents: símptomes, malalties, fàrmacs...

Veiem primer com és un text gold real, i després apliquem el model general.

In [ ]:
# ── Funcions auxiliars ───────────────────────────────────────────────────────


def carregar_gold(nom_fitxer, df):
    """Retorna llista de (inici, fi, etiqueta, text) per a un fitxer."""
    sub = df[df.name == nom_fitxer].copy()
    resultat = []
    for _, fila in sub.iterrows():
        inici, fi = map(int, fila.span.split(","))
        resultat.append((inici, fi, fila.tag, fila.text))
    return resultat


def gold_a_doc(text, gold_list, nlp):
    """Crea un doc spaCy amb les entitats gold com a .ents."""
    doc = nlp.make_doc(text)
    spans = []
    for inici, fi, etiqueta, _ in gold_list:
        span = doc.char_span(inici, fi, label=etiqueta)
        if span is not None:
            spans.append(span)
    doc.ents = spacy.util.filter_spans(spans)
    return doc


COLORS_BIO = {
    "SINTOMA": "#AED6F1",
    "PROCEDIMIENTO": "#F1948A",
    "ENFERMEDAD": "#A9DFBF",
    "FARMACO": "#FAD7A0",
    "SPECIES": "#D7BDE2",
    "HUMANO": "#A3E4D7",
}

In [ ]:
# ── Visualitzem les anotacions gold del primer fitxer ────────────────────────
nom_fitxer = fitxers[0]
text = (TXT_REPLACED / f"{nom_fitxer}.txt").read_text(encoding="utf-8")
gold = carregar_gold(nom_fitxer, df)

print(f"Fitxer: {nom_fitxer}")
print(f"Longitud del text: {len(text)} caràcters")
print(f"Entitats gold: {len(gold)}")
print("\n--- Primers 400 caràcters del text ---")
print(text[:400])

In [ ]:
# ── Visualitzem les entitats gold amb displaCy ───────────────────────────────
doc_gold = gold_a_doc(text, gold, nlp_es)
displacy.render(doc_gold, style="ent", options={"colors": COLORS_BIO})

In [ ]:
# ── Apliquem el model general i comparem ─────────────────────────────────────
doc_es = nlp_es(text)

print("Entitats detectades pel model general (es_core_news_sm):")
print(f"  Total: {len(doc_es.ents)}")
print()
# for ent in doc_es.ents:
#     print(f"  {ent.label_:8s} → '{ent.text.strip()}'")

displacy.render(doc_es, style="ent")

In [ ]:
print("Significat de les categories del model general:")
for label in nlp_es.get_pipe("ner").labels:
    print(f"  {label:<12} → {spacy.explain(label)}")

### Exercici 2

Mira les dues visualitzacions (gold vs. model general) i respon:

1. Quin tipus d'entitats detecta correctament el model general?
2. Quines entitats clíniques **no detecta** en absolut?
3. Hi ha alguna entitat que detecti però que sigui incorrecta (fals positiu)?

## 2. EntityRuler: NER basat en regles

L'`EntityRuler` permet afegir regles a la pipeline d'spaCy. Podem definir patrons basats en text exacte o en atributs lingüístics (PoS, lema, etc.).

### 2.1 Fem un primer model a mà

In [ ]:
# Afegim el ruler abans del NER per prioritzar les nostres regles
nlp_es_ruler = spacy.load("es_core_news_sm")  # copia del model
ruler = nlp_es_ruler.add_pipe("entity_ruler", before="ner")

patrons = [
    # Fàrmacs comuns
    {"label": "FARMACO", "pattern": [{"LOWER": "ibuprofeno"}]},
    {"label": "FARMACO", "pattern": [{"LOWER": "paracetamol"}]},
    {"label": "FARMACO", "pattern": [{"LOWER": "omeprazol"}]},
    {"label": "FARMACO", "pattern": [{"LOWER": "amoxicilina"}]},
    {"label": "FARMACO", "pattern": [{"LOWER": "aspirina"}]},
    # Símptomes
    {"label": "SINTOMA", "pattern": [{"LOWER": "fiebre"}]},
    {"label": "SINTOMA", "pattern": [{"LOWER": "dolor"}, {"LOWER": "de"}, {"LOWER": "cabeza"}]},
    {"label": "SINTOMA", "pattern": [{"LOWER": "náuseas"}]},
    {"label": "SINTOMA", "pattern": [{"LOWER": "disnea"}]},
    {"label": "SINTOMA", "pattern": [{"LOWER": "mareo"}]},
    # Malalties
    {"label": "ENFERMEDAD", "pattern": [{"LOWER": "covid"}]},
    {"label": "ENFERMEDAD", "pattern": [{"LOWER": "covid-19"}]},
    {"label": "ENFERMEDAD", "pattern": [{"LOWER": "hipertensión"}]},
    {"label": "ENFERMEDAD", "pattern": [{"LOWER": "diabetes"}]},
    {"label": "ENFERMEDAD", "pattern": [{"LOWER": "asma"}]},
    {"label": "ENFERMEDAD", "pattern": [{"LOWER": "neumonía"}]},
]

ruler.add_patterns(patrons)

In [ ]:
# Tornem a processar el mateix text
doc2 = nlp_es_ruler(text)

# print("=== Entitats detectades amb les regles ===")
# for ent in doc2.ents:
#     print(f"  {ent.text:<30} {ent.label_}")

displacy.render(doc2, style="ent", options={"colors": COLORS_BIO})

### 2.2. Model amb les anotacions

Estratègia: usarem els **propis textos del gold** com a patrons. Això és una mena de diccionari supervisat: funciona bé per a entitats estables (noms de fàrmacs, malalties conegudes) però té limitacions.

> **Atenció:** Estem usant el corpus gold per construir els patrons i avaluar sobre el mateix corpus. Això és un *oracle* (fer trampes) i no seria vàlid en producció. En un projecte real hauríem de separar dades de train i de test.

In [ ]:
nlp_es_ruler = spacy.load("es_core_news_sm")  # copia del model de nou
ruler = nlp_es_ruler.add_pipe("entity_ruler", before="ner", config={"overwrite_ents": True})

In [ ]:
# Filtrem els casos que apareixen etiquetats almenys 5 vegades
counts = df.text.value_counts()
textos = counts[counts > 5].reset_index().text.tolist()
df_filtrat = df[df.text.isin(textos)][["tag", "text"]].drop_duplicates()

In [ ]:
# Creem els patrons: un per a cada text únic del gold
patrons = []
for _, fila in df_filtrat.iterrows():
    patrons.append({"label": fila.tag, "pattern": [{"LOWER": str(fila.text).strip()}]})
ruler.add_patterns(patrons)

print(f"Patrons afegits: {len(patrons)}")
print("Exemples:")
for p in patrons[:8]:
    print(f"  {p['label']:15s} → '{p['pattern']}'")

In [ ]:
# ── Visualitzem les prediccions del ruler sobre un fitxer ─────────────
doc_ruler = nlp_es_ruler(text)
print(f"Entitats detectades: {len(doc_ruler.ents)}")
displacy.render(doc_ruler, style="ent", options={"colors": COLORS_BIO})

## 3. Pipeline d'anonimització

Una de les aplicacions principals del NER en clínica és l'**anonimització**: substituir dades sensibles (noms, dates, llocs, telèfons) per protegir la privacitat.

CARMEN-I ja ha passat per aquest procés en dues variants:

- `masked` → `[**PROFESION**]`
- `replaced` → `hilandero`

Ara simularem una pipeline d'anonimització i la compararem amb el que fa CARMEN-I.

In [ ]:
# Partim de les anotacions anon
df_anon = pd.read_csv(TSV_REPLACED_ANON, sep="\t")
fitxers_anon = df_anon.name.unique()

In [ ]:
# repetim el procediment anterior, afegim els patrons d'anon al model
counts = df_anon.text.value_counts()
textos = counts[counts > 5].reset_index().text.tolist()
df_anon_filtrat = df_anon[df_anon.text.isin(textos)][["tag", "text"]].drop_duplicates()

patrons = []
for _, fila in df_anon_filtrat.iterrows():
    patrons.append({"label": fila.tag, "pattern": [{"LOWER": str(fila.text).strip()}]})
ruler.add_patterns(patrons)

In [ ]:
# ── Funció d'anonimització ────────────────────────────────────────────────────
tags = df_anon.tag.unique()
ETIQUETES_SENSIBLES = {tag: f"[**{tag}**]" for tag in tags}
# També hi podriem afegir els tags del model original
# PER, LOC, ORG, MISC

# Expressions regulars per a patrons estructurats que el NER sol perdre
PATRONS_REGEX = [
    (r"\b\d{1,2}/\d{1,2}/\d{2,4}\b", "[**FECHAS**]"),  # 12/03/2021
    (r"\b\d{4}-\d{1,2}-\d{1,2}\b", "[**FECHAS**]"),  # 2021-03-12
    (r"\b\d{9}\b", "[**NUMERO_TELEFONO**]"),
]


def anonimitzar(text, doc):
    """
    Pas 1: substitueix entitats detectades pel NER
    Pas 2: substitueix patrons de data i telèfon amb regex
    """
    # -- Pas 1: NER --
    # Anem de dreta a esquerra per no desplaçar els índexs
    entitats = sorted(doc.ents, key=lambda e: e.start_char, reverse=True)
    for ent in entitats:
        if ent.label_ in ETIQUETES_SENSIBLES:
            text = text[: ent.start_char] + f"[**{ent.label_}**]" + text[ent.end_char :]

    # -- Pas 2: regex --
    for patro, placeholder in PATRONS_REGEX:
        text = re.sub(patro, placeholder, text)

    return text


# Apliquem sobre el primer fitxer (versió replaced)

nom_fitxer = fitxers_anon[4]
text = (TXT_REPLACED / f"{nom_fitxer}.txt").read_text(encoding="utf-8")

doc_pred = nlp_es_ruler(text)
text_anon = anonimitzar(text, doc_pred)

print("=== TEXT ORIGINAL (primers 400 car.) ===")
print(text[:400])
print()
print("=== TEXT ANONIMITZAT (primers 400 car.) ===")
print(text_anon[:400])

### Comparació amb la versió masked de CARMEN-I

Ara carreguem el mateix fitxer però de la carpeta `masked`, que és la versió anonimitzada real del corpus. Comparem els dos resultats.

In [ ]:
# ── Comparem la nostra anonimització amb la versió masked ────────────────────
ruta_masked = TXT_MASKED / f"{nom_fitxer}.txt"

text_masked_gold = ruta_masked.read_text(encoding="utf-8")
print("=== CARMEN-I masked (primers 400 car.) ===")
print(text_masked_gold[:400])
print()
print("=== La nostra anonimització (primers 400 car.) ===")
print(text_anon[:400])

### Exercici 3

Compara els dos textos (la nostra anonimització vs. la versió masked de CARMEN-I):

1. Identifica **2 fragments** que el nostre sistema ha anonimitzat però CARMEN-I no (possibles falsos positius).
2. Identifica **2 fragments** que CARMEN-I ha anonimitzat però el nostre sistema ha deixat intactes (falsos negatius perillosos).
3. Se t'acut alguna expressió regular addicional per capturar un dels falsos negatius que has trobat?

## Exercici 4 (extra)

Treballeu sobre un fitxer **diferent** del que hem vist a classe.

### Pas 1: Selecciona un fitxer

Escull un dels fitxers anotats. Indica el nom del fitxer triat.

In [ ]:
# Canvia l'índex per escollir el teu fitxer (qualsevol i > 0)
NOM_ENTREGA = fitxers[___]  # <-- canvia ___ per un nombre

text_entrega = (TXT_REPLACED / f"{NOM_ENTREGA}.txt").read_text(encoding="utf-8")
gold_entrega = carregar_gold(NOM_ENTREGA, df)

print(f"Fitxer seleccionat: {NOM_ENTREGA}")
print(f"Longitud: {len(text_entrega)} caràcters")
print(f"Entitats gold: {len(gold_entrega)}")

### Pas 2: Millora amb EntityRuler

Usa `nlp_ruler` (ja construït a la secció 3) i afegeix **patrons nous** específics per al teu fitxer (mira quines entitats del gold no es detecten i afegeix-les).

In [ ]:
# Mira quines entitats gold no detecta el ruler actual
doc_entrega_ruler = nlp_es_ruler(text_entrega)

gold_set_e = {(i, f, l) for i, f, l, _ in gold_entrega}
pred_set_e = {(ent.start_char, ent.end_char, ent.label_) for ent in doc_entrega_ruler.ents}
fn_exemples = gold_set_e - pred_set_e

print("Entitats gold no detectades (FN):")
for ini, fi, lab in list(fn_exemples):
    print(f"  {lab:15s} → '{text_entrega[ini:fi]}'")

In [ ]:
# Afegeix els teus patrons nous aquí
patrons_entrega = [
    # {"label": "CATEGORIA", "pattern": "text de l'entitat"},
    # ...
]

# Construïm una pipeline amb els patrons nous incorporats
nlp_entrega = spacy.load("es_core_news_sm")
ruler_entrega = ...

doc_entrega_nou = nlp_entrega(text_entrega)

### Pas 3: Anonimització i comparació amb masked

Aplica la pipeline d'anonimització al teu fitxer i compara-la amb la versió masked de CARMEN-I.

In [ ]:
# Anonimitza el teu fitxer
text_entrega_anon = anonimitzar(text_entrega, doc_entrega_nou)

# Carrega la versió masked
ruta_masked_entrega = TXT_MASKED / f"{NOM_ENTREGA}.txt"

print("=== La nostra anonimització (primers 500 car.) ===")
print(text_entrega_anon[:500])
print()

if ruta_masked_entrega.exists():
    text_masked_entrega = ruta_masked_entrega.read_text(encoding="utf-8")
    print("=== CARMEN-I masked (primers 500 car.) ===")
    print(text_masked_entrega[:500])
else:
    print(f"No hi ha versió masked per a {NOM_ENTREGA}")

**Pregunta 3 (reflexió final):** Amb tot el que has vist avui, respon:

- Per a la tasca d'**anonimització clínica**, quin és el risc principal del nostre sistema?
- Quin tipus de tècnica (regles, diccionari, model entrenat) creus que seria necessari per tenir un sistema fiable?
- Per a la detecció de FARMACO o ENFERMEDAD, podria ser suficient un sistema basat en regles i diccionaris? Quan i quan no?